# Global Superstore Exploratory Data Analysis (EDA)

**Dataset:** `Global_Superstore.csv`  
**Project Type:** Portfolio-ready Retail Sales EDA  
**Objective:** Analyze sales performance, profitability drivers, and time-based trends to identify actionable business opportunities.

## Step 1: Problem Statement

Retail organizations need to balance **revenue growth** with **sustainable profit** across products, customer segments, and regions.

### Business Objective
- Identify key drivers of sales and profit.
- Detect underperforming categories, regions, and discount patterns.
- Understand seasonal sales behavior over time.
- Provide data-backed recommendations to improve profitability and planning.

## Step 2: Import Libraries

We import the core Python libraries required for data manipulation and visualization.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization style
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11

## Step 3: Load Dataset

The dataset may include non-UTF8 characters, so we use `latin1` encoding for safe loading.

In [ ]:
df = pd.read_csv('Global_Superstore.csv', encoding='latin1')
print('Dataset loaded successfully!')
df.head()

## Step 4: Data Understanding

In this step, we check data dimensions, schema, data types, and summary statistics to build initial familiarity.

In [ ]:
print('Shape:', df.shape)
print('\nColumns:\n', list(df.columns))
print('\nData Types:\n')
print(df.dtypes)
print('\nSummary Statistics:\n')
display(df.describe(include='all').T.head(24))

## Step 5: Data Cleaning

Data quality is essential before analysis. We will:
1. Check and handle missing values.
2. Remove duplicate records.
3. Convert date columns to datetime format.
4. Validate numeric columns and create clean analysis copy.

In [ ]:
df_clean = df.copy()

# 1) Missing values overview
missing_values = df_clean.isna().sum().sort_values(ascending=False)
print('Missing values (top columns):')
print(missing_values.head(10))

# Postal Code has many missing values for countries where code may not apply.
# Keep as string to avoid numeric coercion issues.
df_clean['Postal Code'] = df_clean['Postal Code'].astype('string')

# 2) Remove duplicates
before = len(df_clean)
df_clean = df_clean.drop_duplicates().copy()
after = len(df_clean)
print(f'\nDuplicates removed: {before - after}')

# 3) Fix date columns
for col in ['Order Date', 'Ship Date']:
    df_clean[col] = pd.to_datetime(df_clean[col], dayfirst=True, errors='coerce')

# 4) Ensure numeric columns are numeric
num_cols = ['Sales', 'Quantity', 'Discount', 'Profit', 'Shipping Cost']
for col in num_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

print('\nCleaned dataset shape:', df_clean.shape)
print('Any nulls in date columns:', df_clean[['Order Date', 'Ship Date']].isna().sum().to_dict())

## Step 6: Univariate Analysis

We analyze the individual distribution of key metrics:
- Sales
- Profit
- Quantity

Histograms help understand spread/skewness, while boxplots highlight outliers.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 14))
metrics = ['Sales', 'Profit', 'Quantity']

for i, metric in enumerate(metrics):
    sns.histplot(df_clean[metric], kde=True, ax=axes[i, 0], color='steelblue')
    axes[i, 0].set_title(f'{metric} Distribution')

    sns.boxplot(x=df_clean[metric], ax=axes[i, 1], color='orange')
    axes[i, 1].set_title(f'{metric} Boxplot')

plt.tight_layout()
plt.show()

print('Key Observation: Sales and Profit are right-skewed with visible outliers; Quantity is concentrated at lower values.')

## Step 7: Bivariate Analysis

Now we examine pairwise relationships:
1. `Sales` vs `Profit`
2. `Category` vs `Sales`
3. `Region` vs `Profit`

This reveals performance differences across dimensions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Sales vs Profit
sns.scatterplot(data=df_clean, x='Sales', y='Profit', alpha=0.4, ax=axes[0])
axes[0].set_title('Sales vs Profit')

# Category vs Sales
cat_sales = df_clean.groupby('Category', as_index=False)['Sales'].sum().sort_values('Sales', ascending=False)
sns.barplot(data=cat_sales, x='Category', y='Sales', ax=axes[1], palette='viridis')
axes[1].set_title('Category vs Total Sales')
axes[1].tick_params(axis='x', rotation=20)

# Region vs Profit
region_profit = df_clean.groupby('Region', as_index=False)['Profit'].sum().sort_values('Profit', ascending=False)
sns.barplot(data=region_profit, x='Region', y='Profit', ax=axes[2], palette='magma')
axes[2].set_title('Region vs Total Profit')
axes[2].tick_params(axis='x', rotation=75)

plt.tight_layout()
plt.show()

print('Observation: Technology leads sales, and regional profitability varies significantly.')

## Step 8: Multivariate Analysis

A correlation heatmap helps identify linear relationships among numerical features and highlights potential drivers of profitability.

In [ ]:
corr_cols = ['Sales', 'Quantity', 'Discount', 'Profit', 'Shipping Cost']
correlation_matrix = df_clean[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap (Numerical Features)')
plt.show()

print('Observation: Discount has a noticeable negative correlation with Profit.')

## Step 9: Time-based Analysis (Monthly and Yearly Trends)

Time analysis helps identify growth trajectory and seasonality, useful for inventory, campaign planning, and forecasting.

In [ ]:
time_df = df_clean.copy()
time_df = time_df.dropna(subset=['Order Date'])
time_df = time_df.sort_values('Order Date')

monthly_sales = time_df.set_index('Order Date').resample('ME')['Sales'].sum()
yearly_sales = time_df.set_index('Order Date').resample('YE')['Sales'].sum()
month_name_sales = time_df.groupby(time_df['Order Date'].dt.month_name())['Sales'].sum()
month_name_sales = month_name_sales.reindex([
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December'
])

fig, axes = plt.subplots(3, 1, figsize=(14, 16))

axes[0].plot(monthly_sales.index, monthly_sales.values, color='tab:blue', linewidth=2)
axes[0].set_title('Monthly Sales Trend')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Sales')

axes[1].plot(yearly_sales.index.year, yearly_sales.values, marker='o', color='tab:green', linewidth=2)
axes[1].set_title('Yearly Sales Trend')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Sales')

sns.barplot(x=month_name_sales.index, y=month_name_sales.values, ax=axes[2], palette='crest')
axes[2].set_title('Seasonality View: Total Sales by Calendar Month')
axes[2].set_xlabel('Month')
axes[2].set_ylabel('Total Sales')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('Best monthly sales point:', monthly_sales.idxmax().strftime('%Y-%m'), '->', round(monthly_sales.max(), 2))
print('Lowest monthly sales point:', monthly_sales.idxmin().strftime('%Y-%m'), '->', round(monthly_sales.min(), 2))

## Step 10: Key Business Insights

Based on analysis of ~51K transactions:

1. Total sales are **$12.64M** with total profit of **$1.47M**, giving an overall margin of about **11.61%**.
2. Sales grew strongly year-over-year from **$2.26M (2011)** to **$4.30M (2014)**, indicating sustained business expansion.
3. **Technology** is the highest-revenue category (~**$4.74M**), making it a primary growth engine.
4. **Tables** show a major profitability issue: high sales but **negative total profit** (~**-$64K**).
5. Discount and profit are negatively correlated (about **-0.32**), suggesting aggressive discounting harms margins.
6. Profit per order is healthy at lower discount bands, but becomes negative around **20%+ discounts**, and deteriorates sharply at higher levels.
7. Regional performance is uneven: some regions (e.g., **North Asia, Central Asia, North**) show strong margins, while others (e.g., **Southeast Asia**) are low-margin.
8. The sales peak month is **Nov 2014 (~$555K)**, while **Feb 2011 (~$91K)** is among the weakest points, confirming strong seasonality.
9. **Home Office** has a slightly higher margin than other segments, although segment-level profitability is generally similar.
10. Top-revenue countries include **United States, Australia, France, China, Germany**, showing concentration that can guide market prioritization.

## Step 11: Conclusion

This EDA reveals that the business is scaling well in sales, but profit quality varies by product mix, discount strategy, and geography. The strongest opportunities are to optimize discount policies, fix loss-making sub-categories (especially Tables), and align inventory/marketing plans with seasonal peaks.

### Recommended Actions
- Set discount guardrails by category and margin thresholds.
- Review pricing, sourcing, and shipping for low-margin sub-categories.
- Prioritize high-margin regions and products in growth campaigns.
- Use monthly seasonality signals for demand forecasting and inventory planning.

This project demonstrates a complete analyst workflow: data understanding, cleaning, visualization, trend analysis, and business storytelling.